# Gold helper — poc SCD functions

Same functions as the LongChau poc repo, loaded with `%run ./_scd` (poc: `from shared_udf import ...`):

| Function | poc file | What it does |
|---|---|---|
| `add_checksums(df, primary_key, exclude)` | `common/hash_utils.py` | `primary_checksum` = md5 of the key columns (columns ending in `id` upper-cased); `record_checksum` = md5 of every other column except metadata, checksums and `exclude` |
| `add_scd_checksum(df, scd_cols)` | `common/hash_utils.py` | `scd_checksum` = md5 of the SCD 2 columns |
| `full_refresh(df, table)` | `common/full_refresh.py` | overwrite the whole table (one atomic commit) |
| `scd_type_1(df, table, primary_key)` | `common/scd_type_1.py` | MERGE on `primary_checksum`, update when `record_checksum` differs |
| `scd_type_2(df, table)` | `common/scd_type_2_new.py` | minor change = update in place, major change = close the current row + insert a new version |
| `get_incremental_dates(from_date, to_date)` | `transform/gold/common/get_incremental_dates.py` | empty widgets → yesterday .. tomorrow (Vietnam) |

Metadata columns, never hashed: `etl_date`, `etl_by`, `current_flag`, `start_date`, `end_date`.

The `*_scd1` notebooks write their MERGE inline so every condition is visible; `scd_type_1` stays here for reuse.

In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, ArrayType
from delta.tables import DeltaTable

METADATA_COLUMNS = ["etl_date", "etl_by", "current_flag", "start_date", "end_date"]
CHECKSUM_COLUMNS = ["primary_checksum", "record_checksum", "scd_checksum"]
MAX_DATE = "9999-12-31 23:59:59"
TZ = "Asia/Ho_Chi_Minh"


def _prepared(df, columns, upper_columns=()):
    """NULL -> '', struct/array -> JSON, everything else -> string; optional upper()."""
    out = []
    for c in columns:
        dtype = df.schema[c].dataType
        e = F.to_json(F.col(c)) if isinstance(dtype, (StructType, ArrayType)) else F.col(c).cast("string")
        e = F.coalesce(e, F.lit(""))
        out.append(F.upper(e) if c in upper_columns else e)
    return out


def add_checksums(df, primary_key, exclude=()):
    """primary_checksum (key) + record_checksum (every non-metadata column minus `exclude`)."""
    skip = set(METADATA_COLUMNS + CHECKSUM_COLUMNS) | set(exclude)
    upper_cols = [c for c in primary_key if c.lower().endswith("id")]
    record_cols = [c for c in df.columns if c not in skip]
    return (df.withColumn("primary_checksum", F.md5(F.concat_ws("||", *_prepared(df, primary_key, upper_cols))))
              .withColumn("record_checksum", F.md5(F.concat_ws("||", *_prepared(df, record_cols, upper_cols)))))


def add_scd_checksum(df, scd_cols, exclude=()):
    """scd_checksum over the SCD 2 columns; '*' = every non-metadata column minus `exclude`."""
    skip = set(METADATA_COLUMNS + CHECKSUM_COLUMNS) | set(exclude)
    cols = [c for c in df.columns if c not in skip] if "*" in scd_cols else list(scd_cols)
    return df.withColumn("scd_checksum", F.md5(F.concat_ws("||", *_prepared(df, cols))))


def full_refresh(df, table):
    """Overwrite the whole table in one commit (poc full_refresh)."""
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)


def get_incremental_dates(from_date, to_date):
    """Empty widgets -> yesterday .. tomorrow (Vietnam). poc uses yesterday .. today; the job runs 3 times a day,
    so today's reads are included too (a rerun writes nothing thanks to the checksum)."""
    if not from_date.strip() and not to_date.strip():
        today = datetime.now(ZoneInfo(TZ)).date()
        return str(today - timedelta(days=1)), str(today + timedelta(days=1))
    return from_date.strip(), to_date.strip()

In [ ]:
def scd_type_1(source_df, table, primary_key, order_col="modified_at"):
    """SCD 1: MERGE on primary_checksum, update only when record_checksum differs.
    order_col: skip source rows older than the target row (not in poc; None = poc behaviour)."""
    cond = "t.record_checksum <> s.record_checksum"
    if order_col:
        cond += f" AND (t.{order_col} IS NULL OR s.{order_col} >= t.{order_col})"
    (DeltaTable.forName(spark, table).alias("t")
        .merge(source_df.alias("s"), "t.primary_checksum = s.primary_checksum")
        .whenMatchedUpdate(condition=cond, set={c: f"s.{c}" for c in source_df.columns if c not in primary_key})
        .whenNotMatchedInsert(values={c: f"s.{c}" for c in source_df.columns})
        .execute())

In [ ]:
def scd_type_2(source_df, table, effective_start_col="created_at", effective_end_col="modified_at", keep=("id",)):
    """SCD 2 (poc new-helper method). Returns the number of minor / major / new rows."""
    target = DeltaTable.forName(spark, table)
    cols = [c for c in source_df.columns if c not in ("current_flag", "start_date", "end_date") and c not in keep]

    cur = (target.toDF().where("current_flag = 'Y'")
           .select("primary_checksum", "record_checksum",
                   F.col("scd_checksum").alias("_t_scd"), F.col(effective_end_col).alias("_t_mod")))

    # rows whose content is not already the current version, and not older than it
    changed = source_df.join(cur.select("record_checksum"), "record_checksum", "left_anti")
    j = changed.join(cur.drop("record_checksum"), "primary_checksum", "left")
    j = j.where(F.col("_t_mod").isNull() | (F.col(effective_end_col) > F.col("_t_mod")))

    minor = j.where("_t_scd IS NOT NULL AND _t_scd = scd_checksum").select(*cols)   # same key, same scd_checksum
    major = j.where("_t_scd IS NOT NULL AND _t_scd <> scd_checksum").select(*cols)  # same key, scd_checksum changed
    new = j.where("_t_scd IS NULL").select(*cols)                                   # key not in the table yet
    counts = {"minor": minor.count(), "major": major.count(), "new": new.count()}

    # 1) minor: update the current row in place
    (target.alias("t")
        .merge(minor.alias("s"), "t.primary_checksum = s.primary_checksum AND t.current_flag = 'Y'")
        .whenMatchedUpdate(set={c: f"s.{c}" for c in cols})
        .execute())

    # 2) major + new in ONE merge: merge_key = key closes the current row, merge_key = NULL inserts the new version
    staged = (
        new.withColumn("merge_key", F.col("primary_checksum"))
           .withColumn("start_date", F.col(effective_start_col).cast("timestamp"))
        .unionByName(major.withColumn("merge_key", F.lit(None).cast("string"))
                          .withColumn("start_date", F.col(effective_end_col).cast("timestamp")))
        .unionByName(major.withColumn("merge_key", F.col("primary_checksum"))
                          .withColumn("start_date", F.lit(None).cast("timestamp")))
        .withColumn("end_date", F.lit(MAX_DATE).cast("timestamp"))
        .withColumn("current_flag", F.lit("Y"))
    )
    (target.alias("t")
        .merge(staged.alias("s"), "t.primary_checksum = s.merge_key AND t.current_flag = 'Y'")
        .whenMatchedUpdate(set={"current_flag": "'N'", "end_date": f"s.{effective_end_col}"})
        .whenNotMatchedInsert(values={c: f"s.{c}" for c in cols + ["start_date", "end_date", "current_flag"]})
        .execute())
    return counts